# Practice 10 – Shortest Paths and Breadth First Search (BFS)

## Introduction to Computer Science for Biologists  
### Shortest paths, BFS, path reconstruction, connectivity, and biological network metrics

In this notebook we will practice concepts from Chapter 6: **Shortest Paths and Breadth First Search (BFS)**.

We will focus on:

- shortest paths in unweighted graphs
- BFS distances
- reconstructing shortest paths
- graph connectivity
- average shortest-path length
- graph diameter
- biological interpretation of shortest paths in networks

The graph is represented as a **binary adjacency matrix**:

- `G[u][v] == 1` means there is an edge from node `u` to node `v`
- `G[u][v] == 0` means there is no edge from `u` to `v`


# Part 0 – Helper Code

Run the following cells before starting the exercises.

You do **not** need to change this code.


In [ ]:
import math
import matplotlib.pyplot as plt


In [ ]:
def draw_graph(G, labels=None, directed=True, title="Graph"):
    """Draw a small graph represented by an adjacency matrix."""

    n = len(G)

    if labels is None:
        labels = list(range(n))

    # Place nodes on a circle
    positions = {}
    for i in range(n):
        angle = 2 * math.pi * i / n
        positions[i] = (math.cos(angle), math.sin(angle))

    plt.figure(figsize=(5, 5))

    # Draw edges
    for u in range(n):
        for v in range(n):
            if G[u][v] == 1:
                x1, y1 = positions[u]
                x2, y2 = positions[v]

                if directed:
                    plt.arrow(x1, y1, x2-x1, y2-y1,
                              length_includes_head=True,
                              head_width=0.04,
                              alpha=0.6)
                else:
                    if u < v:
                        plt.plot([x1, x2], [y1, y2], alpha=0.6)

    # Draw nodes
    for i in range(n):
        x, y = positions[i]
        plt.scatter(x, y, s=600)
        plt.text(x, y, str(labels[i]), ha="center", va="center", fontsize=12)

    plt.title(title)
    plt.axis("off")
    plt.show()


# Part 1 – Example Graph


This is a **directed, unweighted graph**.

The nodes are numbered:

```text
0, 1, 2, 3, 4, 5, 6
```


In [ ]:
G = [
    [0, 0, 1, 1, 0, 0, 0],
    [1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 1, 1],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 1, 0, 0],
]

draw_graph(G, title="Directed graph")


# Part 2 – BFS Distances

The function below is the BFS implementation from the chapter.

It receives:

- `G` – an adjacency matrix
- `source` – the start node

It returns:

- `dists` – a list of distances from `source` to every node

A node that cannot be reached keeps distance `inf`.


In [ ]:
def bfs(G, source):

    inf = float("Inf")
    n = len(G)

    # Initialize all distances to infinity
    dists = [inf] * n

    # The distance from the source to itself is 0
    dists[source] = 0

    # Queue of active nodes
    Q = [source]

    # Main BFS loop
    while len(Q) != 0:

        # FIFO: remove the oldest node
        u = Q.pop(0)

        # Scan all possible neighbors
        for v in range(n):

            # If there is an edge u -> v and v was not discovered yet
            if G[u][v] == 1 and dists[v] == inf:

                # First time reaching v = shortest distance to v
                dists[v] = dists[u] + 1

                # Add v to the queue
                Q.append(v)

    return dists


In [ ]:
# Quick check: BFS from source node 0
print(bfs(G, 0))


Expected output:

```python
[0, 2, 1, 1, 3, 2, 2]
```


# Part 3 – BFS With Previous Nodes

So far, BFS gave us only the **length** of the shortest path.

Now we want to reconstruct the path itself.

To do that, we store an additional list:

```python
prevs
```

`prevs[v]` stores the node that discovered `v` for the first time.

In other words:

```text
prevs[v] = "who discovered v?"
```


In [ ]:
def bfs_with_prevs(G, source):

    inf = float("Inf")
    n = len(G)

    dists = [inf] * n
    dists[source] = 0

    # -1 means: this node was not discovered yet
    prevs = [-1] * n

    Q = [source]

    while len(Q) != 0:

        u = Q.pop(0)

        for v in range(n):

            if G[u][v] == 1 and dists[v] == inf:

                dists[v] = dists[u] + 1
                prevs[v] = u
                Q.append(v)

    return dists, prevs


In [ ]:
dists, prevs = bfs_with_prevs(G, 0)

print("dists:", dists)
print("prevs:", prevs)


Expected output:

```python
dists: [0, 2, 1, 1, 3, 2, 2]
prevs: [-1, 3, 0, 0, 6, 2, 3]
```


# Exercise 1 – Understanding `prevs`

Run `bfs_with_prevs(G, 3)`.

Then answer:

1. What is the value of `prevs[4]`?
2. What does this value mean?
3. What is the value of `prevs[3]`?
4. Why does the source node keep value `-1`?


In [ ]:
# Exercise 2 – your code here


# Part 4 – Reconstructing a Shortest Path

The function `shortest_path(G, source, target)` should return the actual shortest path from `source` to `target`.

Example:

```python
shortest_path(G, 0, 4)
```

should return:

```python
[0, 3, 6, 4]
```

The idea:

1. Run BFS to compute `prevs`
2. Start from the target
3. Move backwards using `prevs`
4. Add each previous node to the beginning of the path


# Exercise 2 – Complete `shortest_path`

Complete the function below.

Hint:

- Start with `path = [target]`
- Use a variable `v = target`
- While `v != source`, replace `v` with `prevs[v]`
- Add each new `v` to the beginning of `path`


In [ ]:
def shortest_path(G, source, target):

    dists, prevs = bfs_with_prevs(G, source)

    # If target was not discovered, there is no path
    if prevs[target] == -1 and target != source:
        return []

    # WRITE YOUR CODE HERE

    return path


In [ ]:
# Check your function

print(shortest_path(G, 0, 4))

assert shortest_path(G, 0, 4) == [0, 3, 6, 4]
print("Correct!")


# Exercise 3 – A Path Through a Required Node

Sometimes we want the shortest path from `source` to `target`, but the path must go through a specific node `middle`.

For example:

```text
source → middle → target
```

We can solve this without changing BFS:

1. Find the shortest path from `source` to `middle`
2. Find the shortest path from `middle` to `target`
3. Join the two paths

Be careful not to repeat the middle node twice.


In [ ]:
def path_through_node(G, source, middle, target):

    # WRITE YOUR CODE HERE

    pass


In [ ]:
# Check your function

print(path_through_node(G, 0, 3, 4))

assert path_through_node(G, 0, 3, 4) == [0, 3, 6, 4]
print("Correct!")


# Part 5 – Connected Components

A **connected component** is a group of nodes that are reachable from each other.

A graph with one connected component is connected.

A graph with more than one connected component is disconnected.

Goal:

Write a function that counts how many connected components exist in an undirected graph.

Hint:

- Keep a list/set of visited nodes
- Start BFS from an unvisited node
- Mark all nodes reached by that BFS as visited
- Each BFS run discovers one connected component


# Exercise 4 – Complete `count_components(G)`

In [ ]:
def count_components(G):

    # WRITE YOUR CODE HERE

    pass


In [ ]:
# Check your function

assert count_components(UG_connected) == 1
assert count_components(UG_disconnected) == 2

print("Correct!")


# Part 6 – Characteristic Path Length

The **characteristic path length** is the average length of the shortest paths between all pairs of nodes.

Intuition:

- Small value → nodes are close to each other
- Large value → the graph is more spread out

For this exercise, assume the graph is connected.

To compute it:

1. Run BFS from every node
2. Collect all finite distances between different nodes
3. Return their average


# Exercise 5 – Complete `avg_dist(G)`

Hint:

Ignore distance from a node to itself (`0`).

Also ignore infinite distances.


In [ ]:
def avg_dist(G):

    # WRITE YOUR CODE HERE

    pass


In [ ]:
# Check your function

result = avg_dist(UG_connected)

print(result)

# In this graph, average distance should be approximately 1.6667
assert round(result, 4) == 1.6667

print("Correct!")


# Part 7 – Graph Diameter

The **diameter** of a graph is the longest shortest path in the graph.

In other words:

```text
diameter = the largest distance between two reachable nodes
```

Intuition:

- It tells us how far apart the farthest nodes are.
- It is a global property of the network.


# Part 8 – Biological Interpretation: Betweenness

**Betweenness centrality** counts how many shortest paths pass through a node.

A node with high betweenness can act as a **bridge** between different regions of a network.

In biological networks, such nodes may be important even if their degree is not very high.


# Exercise 6 – Degree vs Betweenness

A protein interaction network contains two proteins:

| Protein | Degree | Betweenness |
|---|---:|---:|
| A | 50 | 2 |
| B | 4 | 300 |

Answer in words:

1. Which protein is connected to more proteins?
2. Which protein may be more important for information transfer?
3. Why can a low-degree protein still be biologically important?


Write your answer here:

1.  
2.  
3.  


# Final Reflection

Answer briefly:

1. What does BFS compute?
2. Why does BFS find shortest paths in unweighted graphs?
3. What is stored in `dists`?
4. What is stored in `prevs`?
5. What is the difference between average shortest-path length and diameter?
6. Why can shortest paths be useful in biological networks?


Write your answers here:

1.  
2.  
3.  
4.  
5.  
6.  
